# AffectLab validation-fitted multimodal calibration

Reload the frozen context-text and audio checkpoints, export validation posteriors without retraining, fit a modality weight and temperature on validation data only, and evaluate once on each held-out test session. Use a GPU runtime.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from google.colab import auth
PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'affectlab-research-raluca-biras'
TEXT_EXPERIMENT = 'iemocap_benchmark4_context3_deberta_v3_small'
AUDIO_EXPERIMENT = 'iemocap_benchmark4_audio_wav2vec2_base'
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)

In [ ]:
import base64, hashlib, json, os, shutil, sys
from pathlib import Path
from google.colab import userdata
REPO_DIR = Path('/content/emotion-aware-role-play-model')
token = userdata.get('GITHUB_TOKEN')
if not token: raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets.')
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
option = f'http.extraHeader=Authorization: Basic {header}'
url = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
if not REPO_DIR.exists(): subprocess.run(['git', '-c', option, 'clone', url, str(REPO_DIR)], check=True)
else: subprocess.run(['git', '-C', str(REPO_DIR), '-c', option, 'pull', '--ff-only'], check=True)
del token, header, option
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ml.txt'], check=True)

In [ ]:
TEXT_DATA = Path('/content/iemocap-context-data')
AUDIO_DATA = Path('/content/iemocap-audio-data')
AUDIO_ROOT = Path('/content/iemocap-audio')
WORK = Path('/content/iemocap-calibration-work')
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', f'gs://{BUCKET}/data/processed/iemocap-text-v2-context3', str(TEXT_DATA)], check=True)
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', f'gs://{BUCKET}/data/processed/iemocap-text-v1', str(AUDIO_DATA)], check=True)
manifest_path = Path('/content/audio-manifest.json')
bundle_path = Path('/content/iemocap-audio.tar.gz')
audio_gcs = f'gs://{BUCKET}/data/processed/iemocap-audio-benchmark4-v1'
subprocess.run(['gcloud', 'storage', 'cp', f'{audio_gcs}/iemocap-benchmark4-audio-v1.tar.gz.manifest.json', str(manifest_path)], check=True)
subprocess.run(['gcloud', 'storage', 'cp', f'{audio_gcs}/iemocap-benchmark4-audio-v1.tar.gz', str(bundle_path)], check=True)
manifest = json.loads(manifest_path.read_text())
digest = hashlib.sha256()
with bundle_path.open('rb') as handle:
    while chunk := handle.read(16 * 1024 * 1024): digest.update(chunk)
assert digest.hexdigest().upper() == manifest['bundle_sha256']
from ml.preprocessing.iemocap_audio import extract_bundle
extract_bundle(bundle_path, AUDIO_ROOT, manifest['files'])

In [ ]:
for fold in range(1, 6):
    print(f'=== Exporting validation fold {fold} ===')
    for family, experiment, modality, data_root in ((
        'iemocap-text', TEXT_EXPERIMENT, 'text', TEXT_DATA),
        ('iemocap-audio', AUDIO_EXPERIMENT, 'audio', AUDIO_DATA),
    ):
        fold_dir = WORK / modality / f'fold-{fold}'
        fold_dir.mkdir(parents=True, exist_ok=True)
        remote = f'gs://{BUCKET}/runs/{family}/{experiment}/fold-{fold}'
        for filename in ('metrics.json', 'test_predictions.jsonl'):
            subprocess.run(['gcloud', 'storage', 'cp', f'{remote}/{filename}', str(fold_dir / filename)], check=True)
        subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', f'{remote}/model', str(fold_dir / 'model')], check=True)
        command = [sys.executable, '-m', 'ml.evaluation.export_iemocap_validation', '--modality', modality, '--data-root', str(data_root), '--fold-dir', str(fold_dir)]
        if modality == 'audio': command += ['--audio-root', str(AUDIO_ROOT)]
        subprocess.run(command, check=True)
        subprocess.run(['gcloud', 'storage', 'cp', str(fold_dir / 'validation_predictions.jsonl'), f'{remote}/validation_predictions.jsonl'], check=True)
        shutil.rmtree(fold_dir / 'model')
        if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
OUTPUT = WORK / 'validation-fitted-fusion'
subprocess.run([sys.executable, '-m', 'ml.evaluation.calibrate_iemocap_fusion', '--text-dir', str(WORK/'text'), '--audio-dir', str(WORK/'audio'), '--output-dir', str(OUTPUT)], check=True)
result = json.loads((OUTPUT/'summary.json').read_text())
display({'fold_parameters': [{'fold': x['fold'], 'text_weight': x['text_weight'], 'temperature': x['temperature']} for x in result['folds']], 'pooled': result['pooled']})
destination = f'gs://{BUCKET}/runs/iemocap-fusion/iemocap_benchmark4_validation_fitted_fusion'
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', str(OUTPUT), destination], check=True)
print('Uploaded:', destination)

Report the pooled metrics and all five fitted weights and temperatures. Do not select between this model and equal fusion using test ECE alone; the validation-fitted method is the deployable calibration procedure. Keep validation predictions private.